In [3]:
!pip install -q \
    "torch==2.5.1" "torchvision==0.20.1" \
    "timm==1.0.11" \
    "open_clip_torch==2.29.0" \
    "albumentations==1.4.21" \
    "opencv-python-headless==4.10.0.84" \
    "pandas==2.2.3" \
    "numpy==1.26.4" \
    "scikit-learn==1.5.2" \
    "pyarrow==18.0.0" \
    "tqdm==4.67.1"

In [3]:
# ============================================================
# Cell 1: Imports, environment check, and global config
# ============================================================
import os
import sys
import json
import time
import random
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast

import timm
import open_clip
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# ---- Environment sanity check ----
print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 support: {torch.cuda.is_bf16_supported()}")
    print(f"CUDA cap    : {torch.cuda.get_device_capability(0)}")
print(f"timm        : {timm.__version__}")
print(f"open_clip   : {open_clip.__version__}")
print(f"albumentations: {A.__version__}")
print("=" * 60)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Paths (matched to your /dataset/ layout) ----
DATASET_ROOT = Path("/dataset")
CSV_PATH = DATASET_ROOT / "master_v4.csv"
IMAGE_ROOT = DATASET_ROOT  # CSV paths are likely relative to /dataset/
CKPT_DIR = DATASET_ROOT / "checkpoints_loo"
LOG_DIR = DATASET_ROOT / "logs_loo"
RESULTS_DIR = DATASET_ROOT / "results_loo"

for d in [CKPT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Global hyperparameters ----
@dataclass
class Config:
    # Data
    image_size: int = 224
    batch_size: int = 96
    num_workers: int = 12
    val_frac: float = 0.20
    
    # EFB4 — gets the long training
    efb4_total_epochs: int = 18
    efb4_swa_window: int = 5
    efb4_backbone_lr: float = 1e-5
    efb4_head_lr: float = 1e-3
    efb4_warmup_epochs: int = 1
    
    # Swin-T — shorter
    swin_total_epochs: int = 10
    swin_swa_window: int = 3
    swin_backbone_lr: float = 2e-5
    swin_head_lr: float = 1e-3
    swin_warmup_epochs: int = 1
    
    # CLIP ViT-B/16 — shorter
    clip_total_epochs: int = 10
    clip_swa_window: int = 3
    clip_backbone_lr: float = 1e-5
    clip_head_lr: float = 5e-4
    clip_warmup_epochs: int = 1
    
    # Shared training tricks
    label_smoothing: float = 0.05
    grad_clip_norm: float = 1.0
    weight_decay: float = 0.01
    ema_decay: float = 0.9999
    
    # TTA at eval
    tta_views: int = 4
    
    # Mixed precision
    amp_dtype: torch.dtype = torch.bfloat16
    
    # Reproducibility
    seed: int = SEED

CFG = Config()

print("\nCONFIG")
print("=" * 60)
for k, v in asdict(CFG).items():
    print(f"  {k:25s}: {v}")
print("=" * 60)

# ---- Verify data file exists ----
print("\nDATA CHECK")
print("=" * 60)
print(f"Dataset root : {DATASET_ROOT}")
print(f"CSV path     : {CSV_PATH}")
print(f"CSV exists   : {CSV_PATH.exists()}")

if CSV_PATH.exists():
    df_full = pd.read_csv(CSV_PATH)
    print(f"\nTotal rows   : {len(df_full):,}")
    print(f"CSV columns  : {list(df_full.columns)}")
    print(f"\nDtypes:")
    print(df_full.dtypes.to_string())
    print(f"\nFirst 5 rows:")
    print(df_full.head().to_string())
    
    # Try to find dataset/label columns regardless of exact name
    possible_dataset_cols = [c for c in df_full.columns 
                             if c.lower() in ('dataset', 'source', 'dataset_name', 'src')]
    possible_label_cols = [c for c in df_full.columns 
                           if c.lower() in ('label', 'class', 'target', 'y', 'is_fake')]
    possible_path_cols = [c for c in df_full.columns 
                          if c.lower() in ('path', 'image_path', 'filename', 'filepath', 'image', 'file')]
    
    print(f"\nDetected dataset col: {possible_dataset_cols}")
    print(f"Detected label col  : {possible_label_cols}")
    print(f"Detected path col   : {possible_path_cols}")
    
    if possible_dataset_cols:
        col = possible_dataset_cols[0]
        print(f"\nPer-dataset counts ({col}):")
        print(df_full[col].value_counts().to_string())
    
    if possible_label_cols:
        col = possible_label_cols[0]
        print(f"\nLabel distribution ({col}):")
        print(df_full[col].value_counts().to_string())
    
    if possible_label_cols and possible_dataset_cols:
        print(f"\nLabel x Dataset crosstab:")
        print(pd.crosstab(df_full[possible_dataset_cols[0]], 
                          df_full[possible_label_cols[0]]).to_string())
    
    # Check if image paths resolve
    if possible_path_cols:
        sample_path = df_full[possible_path_cols[0]].iloc[0]
        print(f"\nSample path raw     : {sample_path}")
        # Try as-is
        if Path(sample_path).exists():
            print(f"Path exists as-is   : YES")
        # Try relative to /dataset/
        elif (IMAGE_ROOT / sample_path).exists():
            print(f"Path exists relative to /dataset/: YES")
        else:
            print(f"Path NOT found — check IMAGE_ROOT setting")
            print(f"  Tried: {sample_path}")
            print(f"  Tried: {IMAGE_ROOT / sample_path}")
else:
    print("[!] CSV not found at /dataset/master_v4.csv")

print("=" * 60)
print("Cell 1 complete.")

/venv/main/lib/python3.12/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.21). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


ENVIRONMENT CHECK
Python      : 3.12.13
PyTorch     : 2.11.0+cu130
CUDA avail  : True
Device      : NVIDIA H100 80GB HBM3
VRAM        : 85.0 GB
BF16 support: True
CUDA cap    : (9, 0)
timm        : 1.0.11
open_clip   : 2.29.0
albumentations: 1.4.21

CONFIG
  image_size               : 224
  batch_size               : 96
  num_workers              : 12
  val_frac                 : 0.2
  efb4_total_epochs        : 18
  efb4_swa_window          : 5
  efb4_backbone_lr         : 1e-05
  efb4_head_lr             : 0.001
  efb4_warmup_epochs       : 1
  swin_total_epochs        : 10
  swin_swa_window          : 3
  swin_backbone_lr         : 2e-05
  swin_head_lr             : 0.001
  swin_warmup_epochs       : 1
  clip_total_epochs        : 10
  clip_swa_window          : 3
  clip_backbone_lr         : 1e-05
  clip_head_lr             : 0.0005
  clip_warmup_epochs       : 1
  label_smoothing          : 0.05
  grad_clip_norm           : 1.0
  weight_decay             : 0.01
  ema_decay        

In [4]:
import pandas as pd
import os

df = pd.read_csv("/dataset/master_v4.csv")

print("Split values:", df['split'].unique())
print("\nSplit counts:")
print(df['split'].value_counts())

print("\nSplit x dataset:")
print(pd.crosstab(df['dataset'], df['split']))

print("\nSplit x label:")
print(pd.crosstab(df['split'], df['label']))

# Sanity check that paths exist
sample = df.sample(200, random_state=42)
missing = sum(1 for p in sample['path'] if not os.path.exists(p))
print(f"\nMissing files in 200-sample: {missing}/200")

Split values: ['train' 'val' 'test']

Split counts:
split
train    307998
test      60373
val       60372
Name: count, dtype: int64

Split x dataset:
split                         test  train    val
dataset                                         
140k_clean_v1                18460  86138  18458
adversarial                   1500   7000   1500
artifact_clean_v2            11764  54894  11762
celeb_df_v2                   3713  17325   3712
deepfake_and_real_clean_v1   21382  99768  21378
ffpp_c23                      1802  34697   1810
real_and_fake_face_clean_v1    252   1176    252
sd_generated                  1500   7000   1500

Split x label:
label       0       1
split                
test    29609   30764
train  145116  162882
val     29625   30747

Missing files in 200-sample: 0/200


In [5]:
# ============================================================
# Cell 2: Dataset class, augmentations, balanced sampler, loaders
# ============================================================

# ---- The 8 datasets in our LOO sweep ----
ALL_DATASETS = [
    "deepfake_and_real_clean_v1",
    "140k_clean_v1",
    "artifact_clean_v2",
    "ffpp_c23",
    "celeb_df_v2",
    "sd_generated",
    "adversarial",
    "real_and_fake_face_clean_v1",
]

# Datasets that are single-class (all fake) — eval differently
SINGLE_CLASS_DATASETS = {"sd_generated", "adversarial"}

# ---- Load full master CSV once, keep in memory ----
MASTER_DF = pd.read_csv(CSV_PATH)
print(f"Loaded master_v4.csv: {len(MASTER_DF):,} rows")
print(f"Datasets: {sorted(MASTER_DF['dataset'].unique())}")
assert set(MASTER_DF['dataset'].unique()) == set(ALL_DATASETS), \
    "Mismatch between expected datasets and CSV contents"


# ============================================================
# Augmentation pipelines
# ============================================================
# Training: aggressive — geometric + color + compression + cutout
# Val/test: deterministic resize + normalize only
# Why these specific augs: deepfake artifacts often live in JPEG compression
# noise, edge boundaries, and color stats. Augs that destroy these would
# hurt training. So we avoid heavy blur/noise that erases artifacts, and
# use ImageCompression specifically to teach the model robustness across
# compression levels (this is a known win for cross-dataset generalization).

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# CLIP uses its own normalization stats
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)


def build_train_augs(image_size: int, normalize_mean, normalize_std):
    """Training augmentation: aggressive but artifact-preserving."""
    return A.Compose([
        # Resize slightly larger then random crop — adds spatial jitter
        A.SmallestMaxSize(max_size=int(image_size * 1.15), interpolation=cv2.INTER_AREA),
        A.RandomCrop(height=image_size, width=image_size),
        
        # Geometric — symmetric for faces
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.1,
            rotate_limit=10,
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.5,
        ),
        
        # Color/brightness — moderate, don't destroy chroma stats
        A.OneOf([
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=1.0),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=15, p=1.0),
        ], p=0.5),
        
        # Compression robustness — critical for cross-dataset generalization
        A.ImageCompression(quality_lower=60, quality_upper=100, p=0.5),
        
        # Mild blur/noise — low probability so we don't erase artifacts
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MotionBlur(blur_limit=5, p=1.0),
            A.GaussNoise(var_limit=(5.0, 20.0), p=1.0),
        ], p=0.2),
        
        # Cutout / coarse dropout — forces model to use distributed evidence
        A.CoarseDropout(
            max_holes=4, max_height=image_size // 8, max_width=image_size // 8,
            min_holes=1, min_height=image_size // 16, min_width=image_size // 16,
            fill_value=0, p=0.3,
        ),
        
        A.Normalize(mean=normalize_mean, std=normalize_std),
        ToTensorV2(),
    ])


def build_val_augs(image_size: int, normalize_mean, normalize_std):
    """Deterministic eval pipeline."""
    return A.Compose([
        A.SmallestMaxSize(max_size=int(image_size * 1.15), interpolation=cv2.INTER_AREA),
        A.CenterCrop(height=image_size, width=image_size),
        A.Normalize(mean=normalize_mean, std=normalize_std),
        ToTensorV2(),
    ])


def build_tta_augs(image_size: int, normalize_mean, normalize_std):
    """Returns a list of 4 deterministic transforms for test-time augmentation."""
    base_resize = A.SmallestMaxSize(max_size=int(image_size * 1.15), interpolation=cv2.INTER_AREA)
    norm = A.Normalize(mean=normalize_mean, std=normalize_std)
    to_tensor = ToTensorV2()
    
    return [
        # View 1: center crop (canonical)
        A.Compose([base_resize, A.CenterCrop(image_size, image_size), norm, to_tensor]),
        # View 2: hflip + center crop
        A.Compose([base_resize, A.CenterCrop(image_size, image_size),
                   A.HorizontalFlip(p=1.0), norm, to_tensor]),
        # View 3: slight upscale + center crop (different resolution context)
        A.Compose([A.SmallestMaxSize(max_size=int(image_size * 1.30), interpolation=cv2.INTER_AREA),
                   A.CenterCrop(image_size, image_size), norm, to_tensor]),
        # View 4: 5 degree rotation + center crop
        A.Compose([base_resize, A.CenterCrop(image_size, image_size),
                   A.Rotate(limit=(5, 5), p=1.0, border_mode=cv2.BORDER_REFLECT_101),
                   norm, to_tensor]),
    ]


# ============================================================
# Dataset
# ============================================================
class DeepfakeDataset(Dataset):
    """
    Loads images from the master_v4 CSV.
    
    Design notes:
    - We pass in the dataframe slice (already filtered to the rows we want)
      rather than re-filtering inside __init__. This lets one Dataset class
      serve train/val/test/holdout-test without subclassing.
    - We store paths and labels as numpy arrays for fast access (avoids
      pandas indexing overhead in the hot loop).
    - On image read failure, we return a black image rather than crashing —
      LOO sweeps cannot afford to die on one bad file 14 hours in.
    """
    def __init__(self, df: pd.DataFrame, transform, return_index: bool = False):
        self.paths = df['path'].values
        self.labels = df['label'].values.astype(np.float32)
        self.datasets = df['dataset'].values  # for diagnostic logging
        self.transform = transform
        self.return_index = return_index
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]
        
        # cv2 is faster than PIL for jpeg decoding and integrates with albumentations
        img = cv2.imread(path)
        if img is None:
            # Fallback: black image, log it
            img = np.zeros((CFG.image_size, CFG.image_size, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        out = self.transform(image=img)
        tensor = out['image']
        
        if self.return_index:
            return tensor, label, idx
        return tensor, label


# ============================================================
# Composite balanced sampler
# ============================================================
def build_balanced_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    """
    Builds a WeightedRandomSampler that simultaneously balances:
      - real/fake (50/50 in expectation)
      - source dataset (uniform across the 7 training datasets)
    
    The composite weight for sample i is:
        w_i = (1 / count_of_label_i) * (1 / count_of_dataset_i)
    
    This way each (label, dataset) cell gets seen proportionally,
    preventing the model from being dominated by 140k_clean_v1 reals
    or deepfake_and_real_clean_v1 fakes.
    """
    label_counts = df['label'].value_counts().to_dict()
    dataset_counts = df['dataset'].value_counts().to_dict()
    
    label_w = df['label'].map(lambda l: 1.0 / label_counts[l]).values
    dataset_w = df['dataset'].map(lambda d: 1.0 / dataset_counts[d]).values
    
    weights = label_w * dataset_w
    weights = weights / weights.sum()  # normalize for stability
    
    return WeightedRandomSampler(
        weights=torch.from_numpy(weights).double(),
        num_samples=len(df),
        replacement=True,
    )


# ============================================================
# Dataloader factory for one holdout
# ============================================================
def build_holdout_loaders(
    holdout_dataset: str,
    normalize_mean=IMAGENET_MEAN,
    normalize_std=IMAGENET_STD,
    val_frac: float = 0.20,
    seed: int = 42,
) -> Dict:
    """
    Returns train_loader, val_loader, test_loader for a single LOO holdout.
    
    Logic:
    - Train pool = all rows where dataset != holdout
    - 80/20 stratified split of train pool by (dataset, label) to preserve balance
    - Test = all rows where dataset == holdout (whole dataset, ignoring 'split' column)
    """
    # Split master into train pool and held-out test
    train_pool_df = MASTER_DF[MASTER_DF['dataset'] != holdout_dataset].reset_index(drop=True)
    test_df = MASTER_DF[MASTER_DF['dataset'] == holdout_dataset].reset_index(drop=True)
    
    # Stratified 80/20 split — stratify by (dataset, label) composite
    # so each training dataset contributes proportionally to val
    rng = np.random.RandomState(seed)
    train_pool_df['_strat'] = train_pool_df['dataset'] + '_' + train_pool_df['label'].astype(str)
    
    val_indices = []
    train_indices = []
    for stratum, group in train_pool_df.groupby('_strat'):
        idx = group.index.values
        rng.shuffle(idx)
        n_val = int(round(len(idx) * val_frac))
        val_indices.extend(idx[:n_val])
        train_indices.extend(idx[n_val:])
    
    train_df = train_pool_df.loc[train_indices].drop(columns='_strat').reset_index(drop=True)
    val_df = train_pool_df.loc[val_indices].drop(columns='_strat').reset_index(drop=True)
    
    # Build augs
    train_aug = build_train_augs(CFG.image_size, normalize_mean, normalize_std)
    val_aug = build_val_augs(CFG.image_size, normalize_mean, normalize_std)
    
    # Datasets
    train_ds = DeepfakeDataset(train_df, train_aug)
    val_ds = DeepfakeDataset(val_df, val_aug)
    test_ds = DeepfakeDataset(test_df, val_aug, return_index=True)  # need indices for per-image logging
    
    # Sampler — only on train
    train_sampler = build_balanced_sampler(train_df)
    
    # Loaders
    common_kwargs = dict(
        num_workers=CFG.num_workers,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )
    
    train_loader = DataLoader(
        train_ds, batch_size=CFG.batch_size, sampler=train_sampler,
        drop_last=True, **common_kwargs,
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
        drop_last=False, **common_kwargs,
    )
    test_loader = DataLoader(
        test_ds, batch_size=CFG.batch_size * 2, shuffle=False,
        drop_last=False, **common_kwargs,
    )
    
    return {
        'train_loader': train_loader,
        'val_loader': val_loader,
        'test_loader': test_loader,
        'train_df': train_df,
        'val_df': val_df,
        'test_df': test_df,
    }


# ============================================================
# Smoke test: build loaders for the smallest holdout and load one batch
# ============================================================
print("\n" + "=" * 60)
print("SMOKE TEST: building loaders for 'real_and_fake_face_clean_v1'")
print("=" * 60)

t0 = time.time()
loaders = build_holdout_loaders("real_and_fake_face_clean_v1")
t_build = time.time() - t0
print(f"Loader build time: {t_build:.2f}s")

print(f"\nTrain df    : {len(loaders['train_df']):,} rows")
print(f"Val df      : {len(loaders['val_df']):,} rows")
print(f"Test df     : {len(loaders['test_df']):,} rows")
print(f"Train batches: {len(loaders['train_loader'])}")
print(f"Val batches  : {len(loaders['val_loader'])}")
print(f"Test batches : {len(loaders['test_loader'])}")

# Verify per-dataset balance in train
print(f"\nTrain df dataset distribution:")
print(loaders['train_df']['dataset'].value_counts().to_string())

print(f"\nTrain df label distribution:")
print(loaders['train_df']['label'].value_counts().to_string())

# Load one training batch and time it (with sampler -> tests balanced sampling)
print(f"\nLoading one training batch...")
t0 = time.time()
batch = next(iter(loaders['train_loader']))
t_first = time.time() - t0
imgs, lbls = batch
print(f"  First batch time   : {t_first:.2f}s (includes worker startup)")
print(f"  Image tensor shape : {imgs.shape}")
print(f"  Image dtype        : {imgs.dtype}")
print(f"  Image value range  : [{imgs.min():.3f}, {imgs.max():.3f}]")
print(f"  Label shape        : {lbls.shape}")
print(f"  Label distribution : real={int((lbls == 0).sum())}, fake={int((lbls == 1).sum())}")

# Time a few more batches without worker startup overhead
print(f"\nTiming 5 more batches (steady state)...")
t0 = time.time()
loader_iter = iter(loaders['train_loader'])
next(loader_iter)  # warmup
for _ in range(5):
    next(loader_iter)
t_per_batch = (time.time() - t0) / 6
print(f"  Avg per batch      : {t_per_batch * 1000:.1f}ms")
print(f"  Throughput         : {CFG.batch_size / t_per_batch:.0f} imgs/sec (CPU side)")

# One test batch (with index return)
print(f"\nLoading one test batch...")
test_batch = next(iter(loaders['test_loader']))
t_imgs, t_lbls, t_idx = test_batch
print(f"  Test img shape     : {t_imgs.shape}")
print(f"  Test idx shape     : {t_idx.shape}")
print(f"  First 5 indices    : {t_idx[:5].tolist()}")

print("\nCell 2 complete.")

Loaded master_v4.csv: 428,743 rows
Datasets: ['140k_clean_v1', 'adversarial', 'artifact_clean_v2', 'celeb_df_v2', 'deepfake_and_real_clean_v1', 'ffpp_c23', 'real_and_fake_face_clean_v1', 'sd_generated']

SMOKE TEST: building loaders for 'real_and_fake_face_clean_v1'
Loader build time: 0.52s

Train df    : 341,649 rows
Val df      : 85,414 rows
Test df     : 1,680 rows
Train batches: 3558
Val batches  : 445
Test batches : 9

Train df dataset distribution:
dataset
deepfake_and_real_clean_v1    114022
140k_clean_v1                  98444
artifact_clean_v2              62736
ffpp_c23                       30647
celeb_df_v2                    19800
adversarial                     8000
sd_generated                    8000

Train df label distribution:
label
1    178842
0    162807

Loading one training batch...
  First batch time   : 1.57s (includes worker startup)
  Image tensor shape : torch.Size([96, 3, 224, 224])
  Image dtype        : torch.float32
  Image value range  : [-2.118, 2.640]

In [6]:
# Tweak loader settings and re-time
import time

# Rebuild with more workers
CFG.num_workers = 20  # was 12

loaders = build_holdout_loaders("real_and_fake_face_clean_v1")

# Warmup
loader_iter = iter(loaders['train_loader'])
next(loader_iter)

# Time 10 batches at steady state
t0 = time.time()
for _ in range(10):
    next(loader_iter)
t_per_batch = (time.time() - t0) / 10
print(f"With num_workers=20:")
print(f"  Avg per batch : {t_per_batch * 1000:.1f}ms")
print(f"  Throughput    : {CFG.batch_size / t_per_batch:.0f} imgs/sec")
print(f"  Per epoch est : {len(loaders['train_loader']) * t_per_batch / 60:.1f} min (CPU side)")

With num_workers=20:
  Avg per batch : 29.7ms
  Throughput    : 3229 imgs/sec
  Per epoch est : 1.8 min (CPU side)


In [7]:
import multiprocessing
import os

print(f"CPU count        : {multiprocessing.cpu_count()}")
print(f"CPU affinity     : {len(os.sched_getaffinity(0))} cores available to us")

# Check how big a typical image actually is
import cv2
sample_paths = MASTER_DF.sample(10, random_state=1)['path'].tolist()
print(f"\nSample image sizes:")
for p in sample_paths:
    img = cv2.imread(p)
    if img is not None:
        size_kb = os.path.getsize(p) / 1024
        print(f"  {img.shape[0]}x{img.shape[1]}  {size_kb:6.1f} KB  {p.split('/')[-3]}")

CPU count        : 208
CPU affinity     : 208 cores available to us

Sample image sizes:
  224x224    10.9 KB  celeb_df_v2
  224x224    13.9 KB  test
  224x224    82.7 KB  sd_generated
  224x224    18.7 KB  train
  224x224    13.6 KB  test
  224x224    10.7 KB  train
  224x224    15.3 KB  train
  224x224    16.0 KB  test
  224x224    16.4 KB  train
  224x224    17.0 KB  val


In [11]:
# ============================================================
# Cell 3 (combined): fixed model factories + smoke test
# ============================================================
from copy import deepcopy

# ---- Wrapper to squeeze timm output from [B,1] to [B] ----
class TimmBinaryWrapper(nn.Module):
    def __init__(self, timm_model):
        super().__init__()
        self.model = timm_model
    def forward(self, x):
        return self.model(x).squeeze(-1)


# ---- EFB4 ----
def build_efb4(pretrained: bool = True):
    base = timm.create_model(
        'tf_efficientnet_b4',
        pretrained=pretrained,
        num_classes=1,
        drop_rate=0.3,
        drop_path_rate=0.2,
    )
    backbone_params, head_params = [], []
    for name, p in base.named_parameters():
        if 'classifier' in name:
            head_params.append(p)
        else:
            backbone_params.append(p)
    model = TimmBinaryWrapper(base)
    return model, backbone_params, head_params


# ---- Swin-T ----
def build_swin_t(pretrained: bool = True):
    base = timm.create_model(
        'swin_tiny_patch4_window7_224',
        pretrained=pretrained,
        num_classes=1,
        drop_rate=0.1,
        drop_path_rate=0.2,
    )
    backbone_params, head_params = [], []
    for name, p in base.named_parameters():
        if 'head' in name:
            head_params.append(p)
        else:
            backbone_params.append(p)
    model = TimmBinaryWrapper(base)
    return model, backbone_params, head_params


# ---- CLIP ViT-B/16 ----
class CLIPViTBinary(nn.Module):
    def __init__(self, pretrained: bool = True, dropout: float = 0.2):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(
            'ViT-B-16',
            pretrained='openai' if pretrained else None,
        )
        self.visual = clip_model.visual
        feat_dim = self.visual.output_dim
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        feats = self.visual(x)
        return self.head(feats).squeeze(-1)


def build_clip_vit_b16(pretrained: bool = True):
    model = CLIPViTBinary(pretrained=pretrained)
    backbone_params = list(model.visual.parameters())
    head_params = list(model.head.parameters())
    return model, backbone_params, head_params


# ---- Freezing helpers ----
def freeze_backbone(backbone_params):
    for p in backbone_params:
        p.requires_grad = False

def unfreeze_backbone(backbone_params):
    for p in backbone_params:
        p.requires_grad = True


# ---- Optimizer + scheduler factory ----
def build_optimizer_and_scheduler(
    backbone_params, head_params,
    backbone_lr, head_lr,
    weight_decay, total_epochs, steps_per_epoch, warmup_epochs=1,
):
    optimizer = torch.optim.AdamW(
        [
            {'params': backbone_params, 'lr': backbone_lr, 'name': 'backbone'},
            {'params': head_params, 'lr': head_lr, 'name': 'head'},
        ],
        weight_decay=weight_decay,
        betas=(0.9, 0.999),
        eps=1e-8,
    )
    
    total_steps = total_epochs * steps_per_epoch
    warmup_steps = warmup_epochs * steps_per_epoch
    
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
        progress = min(1.0, progress)
        return 0.01 + 0.99 * 0.5 * (1 + np.cos(np.pi * progress))
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


# ---- EMA ----
class ModelEMA:
    def __init__(self, model, decay=0.9999):
        self.ema = deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad = False
        self.decay = decay
    
    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)
            else:
                v.copy_(msd[k])
    
    def state_dict(self):
        return self.ema.state_dict()


# ============================================================
# Smoke test
# ============================================================
print("=" * 60)
print("MODEL SMOKE TEST")
print("=" * 60)

sample_batch = next(iter(loaders['val_loader']))
sample_imgs = sample_batch[0][:CFG.batch_size].to(DEVICE)
sample_lbls = sample_batch[1][:CFG.batch_size].to(DEVICE)
print(f"Sample batch: {sample_imgs.shape} on {sample_imgs.device}")
print(f"Sample labels: {sample_lbls.shape}\n")

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def smoke_test_model(name, model, backbone_params, head_params, lr_b, lr_h, n_epochs):
    model = model.to(DEVICE)
    total, trainable = count_params(model)
    print(f"--- {name} ---")
    print(f"  Total params      : {total / 1e6:.1f}M")
    print(f"  Trainable params  : {trainable / 1e6:.1f}M")
    print(f"  Backbone params   : {sum(p.numel() for p in backbone_params) / 1e6:.1f}M (lr={lr_b})")
    print(f"  Head params       : {sum(p.numel() for p in head_params) / 1e6:.3f}M (lr={lr_h})")
    
    opt, sched = build_optimizer_and_scheduler(
        backbone_params, head_params, lr_b, lr_h,
        weight_decay=CFG.weight_decay,
        total_epochs=n_epochs,
        steps_per_epoch=len(loaders['train_loader']),
        warmup_epochs=1,
    )
    
    criterion = nn.BCEWithLogitsLoss()
    
    # Verify shape compatibility BEFORE timing
    with autocast(dtype=CFG.amp_dtype):
        out = model(sample_imgs)
    print(f"  Output shape      : {out.shape} (expected [{CFG.batch_size}])")
    assert out.shape == sample_lbls.shape, f"Shape mismatch: {out.shape} vs {sample_lbls.shape}"
    
    # Warmup pass
    with autocast(dtype=CFG.amp_dtype):
        out = model(sample_imgs)
        loss = criterion(out, sample_lbls)
    loss.backward()
    opt.zero_grad()
    
    # Time 5 forward+backward passes
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(5):
        opt.zero_grad()
        with autocast(dtype=CFG.amp_dtype):
            out = model(sample_imgs)
            loss = criterion(out, sample_lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip_norm)
        opt.step()
        sched.step()
    torch.cuda.synchronize()
    per_step = (time.time() - t0) / 5
    
    print(f"  Loss              : {loss.item():.4f}")
    print(f"  Per-step time     : {per_step * 1000:.1f}ms")
    print(f"  Throughput        : {CFG.batch_size / per_step:.0f} imgs/sec (GPU)")
    
    n_batches = len(loaders['train_loader'])
    epoch_time = n_batches * per_step
    total_time = epoch_time * n_epochs
    print(f"  Est epoch         : {epoch_time / 60:.1f} min ({n_batches} batches)")
    print(f"  Est total ({n_epochs}ep) : {total_time / 60:.1f} min")
    
    ema = ModelEMA(model, decay=CFG.ema_decay)
    ema.update(model)
    print(f"  EMA wrapper       : OK")
    
    mem_alloc = torch.cuda.memory_allocated() / 1e9
    mem_reserved = torch.cuda.memory_reserved() / 1e9
    print(f"  GPU mem allocated : {mem_alloc:.2f} GB")
    print(f"  GPU mem reserved  : {mem_reserved:.2f} GB")
    print()
    
    del model, opt, sched, ema, out, loss
    torch.cuda.empty_cache()
    
    return per_step, epoch_time, total_time


# Run all three
model_efb4, bp, hp = build_efb4()
efb4_step, efb4_epoch, efb4_total = smoke_test_model(
    "EfficientNet-B4", model_efb4, bp, hp,
    CFG.efb4_backbone_lr, CFG.efb4_head_lr, CFG.efb4_total_epochs,
)

model_swin, bp, hp = build_swin_t()
swin_step, swin_epoch, swin_total = smoke_test_model(
    "Swin-T", model_swin, bp, hp,
    CFG.swin_backbone_lr, CFG.swin_head_lr, CFG.swin_total_epochs,
)

model_clip, bp, hp = build_clip_vit_b16()
clip_step, clip_epoch, clip_total = smoke_test_model(
    "CLIP ViT-B/16", model_clip, bp, hp,
    CFG.clip_backbone_lr, CFG.clip_head_lr, CFG.clip_total_epochs,
)

# Time estimates
print("=" * 60)
print("PER-HOLDOUT TIME ESTIMATE")
print("=" * 60)
total_seq = efb4_total + swin_total + clip_total
print(f"  EFB4   : {efb4_total / 60:5.1f} min")
print(f"  Swin-T : {swin_total / 60:5.1f} min")
print(f"  CLIP   : {clip_total / 60:5.1f} min")
print(f"  Total  : {total_seq / 60:5.1f} min ({total_seq / 3600:.2f} hours)")
print(f"\nFor 8 holdouts (sequential): {8 * total_seq / 3600:.1f} hours")
print(f"Cost @ $1.75/hr            : ${8 * total_seq / 3600 * 1.75:.2f}")
print()
print("With Swin+CLIP parallel after EFB4:")
parallel = efb4_total + max(swin_total, clip_total)
print(f"  Per holdout: {parallel / 60:.1f} min")
print(f"  8 holdouts : {8 * parallel / 3600:.1f} hours")
print(f"  Cost       : ${8 * parallel / 3600 * 1.75:.2f}")

print("\nCell 3 complete.")

MODEL SMOKE TEST
Sample batch: torch.Size([96, 3, 224, 224]) on cuda:0
Sample labels: torch.Size([96])



--- EfficientNet-B4 ---
  Total params      : 17.6M
  Trainable params  : 17.6M
  Backbone params   : 17.5M (lr=1e-05)
  Head params       : 0.002M (lr=0.001)
  Output shape      : torch.Size([96]) (expected [96])
  Loss              : 1.2415
  Per-step time     : 180.9ms
  Throughput        : 531 imgs/sec (GPU)
  Est epoch         : 10.7 min (3558 batches)
  Est total (18ep) : 193.1 min
  EMA wrapper       : OK
  GPU mem allocated : 11.46 GB
  GPU mem reserved  : 23.41 GB



model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

--- Swin-T ---
  Total params      : 27.5M
  Trainable params  : 27.5M
  Backbone params   : 27.5M (lr=2e-05)
  Head params       : 0.001M (lr=0.001)
  Output shape      : torch.Size([96]) (expected [96])
  Loss              : 0.7879
  Per-step time     : 54.2ms
  Throughput        : 1772 imgs/sec (GPU)
  Est epoch         : 3.2 min (3558 batches)
  Est total (10ep) : 32.1 min
  EMA wrapper       : OK
  GPU mem allocated : 11.80 GB
  GPU mem reserved  : 19.26 GB



open_clip_model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

--- CLIP ViT-B/16 ---
  Total params      : 86.3M
  Trainable params  : 86.3M
  Backbone params   : 86.2M (lr=1e-05)
  Head params       : 0.133M (lr=0.0005)
  Output shape      : torch.Size([96]) (expected [96])
  Loss              : 0.6216
  Per-step time     : 53.0ms
  Throughput        : 1812 imgs/sec (GPU)
  Est epoch         : 3.1 min (3558 batches)
  Est total (10ep) : 31.4 min
  EMA wrapper       : OK
  GPU mem allocated : 13.21 GB
  GPU mem reserved  : 20.68 GB

PER-HOLDOUT TIME ESTIMATE
  EFB4   : 193.1 min
  Swin-T :  32.1 min
  CLIP   :  31.4 min
  Total  : 256.6 min (4.28 hours)

For 8 holdouts (sequential): 34.2 hours
Cost @ $1.75/hr            : $59.88

With Swin+CLIP parallel after EFB4:
  Per holdout: 225.2 min
  8 holdouts : 30.0 hours
  Cost       : $52.55

Cell 3 complete.


In [12]:
# ============================================================
# EFB4 speed diagnostic — test channels_last and torch.compile
# ============================================================
print("=" * 60)
print("EFB4 SPEED FIXES")
print("=" * 60)

def time_efb4(label, use_channels_last=False, use_compile=False, n_iters=10):
    """Build EFB4 and time forward+backward with given options."""
    torch.cuda.empty_cache()
    
    base = timm.create_model(
        'tf_efficientnet_b4',
        pretrained=True,
        num_classes=1,
        drop_rate=0.3,
        drop_path_rate=0.2,
    )
    model = TimmBinaryWrapper(base).to(DEVICE)
    
    if use_channels_last:
        model = model.to(memory_format=torch.channels_last)
    
    if use_compile:
        print(f"  Compiling (this takes 30-60s)...")
        model = torch.compile(model, mode='reduce-overhead')
    
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    criterion = nn.BCEWithLogitsLoss()
    
    # Prepare batch in correct format
    imgs = sample_imgs.clone()
    if use_channels_last:
        imgs = imgs.to(memory_format=torch.channels_last)
    
    # Warmup (compile needs several passes to stabilize)
    n_warmup = 5 if use_compile else 2
    for _ in range(n_warmup):
        opt.zero_grad()
        with autocast(dtype=CFG.amp_dtype):
            out = model(imgs)
            loss = criterion(out, sample_lbls)
        loss.backward()
        opt.step()
    
    # Time
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_iters):
        opt.zero_grad()
        with autocast(dtype=CFG.amp_dtype):
            out = model(imgs)
            loss = criterion(out, sample_lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    torch.cuda.synchronize()
    per_step = (time.time() - t0) / n_iters
    
    mem_alloc = torch.cuda.memory_allocated() / 1e9
    
    print(f"  {label}")
    print(f"    Per-step    : {per_step * 1000:.1f}ms")
    print(f"    Throughput  : {CFG.batch_size / per_step:.0f} imgs/sec")
    print(f"    18-ep total : {18 * 3558 * per_step / 60:.1f} min")
    print(f"    Mem         : {mem_alloc:.2f} GB")
    print()
    
    del model, opt
    torch.cuda.empty_cache()
    return per_step

# Baseline (what we already measured)
print("Baseline (NCHW, no compile):")
t_base = time_efb4("baseline", use_channels_last=False, use_compile=False)

# Fix 1: channels_last
print("Fix 1: channels_last memory format:")
t_cl = time_efb4("channels_last", use_channels_last=True, use_compile=False)

# Fix 4: torch.compile
print("Fix 4: torch.compile only:")
t_comp = time_efb4("compile", use_channels_last=False, use_compile=True)

# Both
print("Fix 1+4: channels_last + torch.compile:")
t_both = time_efb4("both", use_channels_last=True, use_compile=True)

print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  Baseline       : {t_base*1000:.1f}ms  (1.00x)")
print(f"  channels_last  : {t_cl*1000:.1f}ms  ({t_base/t_cl:.2f}x)")
print(f"  compile        : {t_comp*1000:.1f}ms  ({t_base/t_comp:.2f}x)")
print(f"  both           : {t_both*1000:.1f}ms  ({t_base/t_both:.2f}x)")

best = min(t_base, t_cl, t_comp, t_both)
saved_per_holdout = (t_base - best) * 3558 * 18 / 60
print(f"\nBest config saves {saved_per_holdout:.1f} min per holdout")
print(f"Across 8 holdouts: {saved_per_holdout * 8 / 60:.1f} hours saved")
print(f"Cost saved: ${saved_per_holdout * 8 / 60 * 1.75:.2f}")

EFB4 SPEED FIXES
Baseline (NCHW, no compile):
  baseline
    Per-step    : 125.9ms
    Throughput  : 762 imgs/sec
    18-ep total : 134.4 min
    Mem         : 12.52 GB

Fix 1: channels_last memory format:
  channels_last
    Per-step    : 63.3ms
    Throughput  : 1517 imgs/sec
    18-ep total : 67.5 min
    Mem         : 12.52 GB

Fix 4: torch.compile only:
  Compiling (this takes 30-60s)...
  compile
    Per-step    : 43.4ms
    Throughput  : 2210 imgs/sec
    18-ep total : 46.4 min
    Mem         : 12.38 GB

Fix 1+4: channels_last + torch.compile:
  Compiling (this takes 30-60s)...
  both
    Per-step    : 43.4ms
    Throughput  : 2213 imgs/sec
    18-ep total : 46.3 min
    Mem         : 12.38 GB

SUMMARY
  Baseline       : 125.9ms  (1.00x)
  channels_last  : 63.3ms  (1.99x)
  compile        : 43.4ms  (2.90x)
  both           : 43.4ms  (2.90x)

Best config saves 88.1 min per holdout
Across 8 holdouts: 11.8 hours saved
Cost saved: $20.56


In [15]:
# ============================================================
# Cell 4: Single-model training loop with EMA + SWA
# ============================================================
import json
from copy import deepcopy
from collections import defaultdict


# ============================================================
# Manual SWA — operates on state_dicts to avoid torch.compile/deepcopy issues
# ============================================================
class ManualSWA:
    """
    Stochastic Weight Averaging implemented as a running mean of state_dicts.
    
    Why manual instead of torch.optim.swa_utils.AveragedModel:
    AveragedModel deepcopies the model, which breaks for torch.compile-wrapped
    models because OptimizedModule isn't always deepcopy-safe. By operating
    directly on state_dicts (which are just OrderedDicts of tensors), we sidestep
    the issue entirely.
    
    Usage:
        swa = ManualSWA()
        for epoch in swa_window:
            train_one_epoch(...)
            swa.update(model)  # accumulates this epoch's weights
        swa_state = swa.get_averaged_state_dict()
    """
    def __init__(self):
        self.n_averaged = 0
        self.averaged_state = None
    
    @torch.no_grad()
    def update(self, model):
        # Strip _orig_mod. prefix from compiled models
        sd = self._clean_state_dict(model.state_dict())
        
        if self.averaged_state is None:
            # First snapshot
            self.averaged_state = {k: v.detach().clone().float() for k, v in sd.items()
                                   if v.dtype.is_floating_point}
            # Non-float buffers: copy directly
            self.non_float_buffers = {k: v.detach().clone() for k, v in sd.items()
                                      if not v.dtype.is_floating_point}
            self.n_averaged = 1
        else:
            # Running mean: avg_new = avg_old + (new - avg_old) / (n+1)
            self.n_averaged += 1
            for k, v_avg in self.averaged_state.items():
                v_new = sd[k].detach().float()
                v_avg.add_((v_new - v_avg) / self.n_averaged)
    
    def get_averaged_state_dict(self):
        """Returns a state_dict ready to load into a model."""
        result = {}
        for k, v in self.averaged_state.items():
            result[k] = v.clone()
        for k, v in self.non_float_buffers.items():
            result[k] = v.clone()
        return result
    
    @staticmethod
    def _clean_state_dict(sd):
        """Strip _orig_mod. prefix that torch.compile adds."""
        return {k.replace('_orig_mod.', ''): v for k, v in sd.items()}


# ============================================================
# EMA — updated to handle torch.compile
# ============================================================
class CompileSafeEMA:
    """
    EMA that works with torch.compile-wrapped models.
    
    For compiled models, we deepcopy the underlying _orig_mod and apply EMA
    updates to that. This avoids deepcopy errors on OptimizedModule.
    """
    def __init__(self, model, decay=0.9999):
        # Get the underlying module if compiled
        base_model = model._orig_mod if hasattr(model, '_orig_mod') else model
        self.ema = deepcopy(base_model).eval()
        for p in self.ema.parameters():
            p.requires_grad = False
        self.decay = decay
    
    @torch.no_grad()
    def update(self, model):
        # Get clean state dict from current model
        msd = self._clean_state_dict(model.state_dict())
        ema_sd = self.ema.state_dict()
        
        for k, v_ema in ema_sd.items():
            if v_ema.dtype.is_floating_point and k in msd:
                v_ema.mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)
            elif k in msd:
                v_ema.copy_(msd[k])
    
    def state_dict(self):
        return self.ema.state_dict()
    
    @staticmethod
    def _clean_state_dict(sd):
        return {k.replace('_orig_mod.', ''): v for k, v in sd.items()}


# ============================================================
# Per-epoch helpers
# ============================================================
@torch.no_grad()
def evaluate_model(model, loader, criterion, device, desc="val"):
    """
    Run model on a loader, return aggregate metrics.
    
    Returns dict with: loss, acc, auroc, n_samples
    Uses bf16 autocast for speed (matches training precision).
    """
    model.eval()
    
    all_logits = []
    all_labels = []
    total_loss = 0.0
    n_samples = 0
    
    for batch in loader:
        # Handle both 2-tuple and 3-tuple batches (test loader has indices)
        if len(batch) == 3:
            imgs, lbls, _ = batch
        else:
            imgs, lbls = batch
        
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        
        with autocast(dtype=CFG.amp_dtype):
            logits = model(imgs)
            loss = criterion(logits, lbls)
        
        total_loss += loss.item() * lbls.size(0)
        n_samples += lbls.size(0)
        all_logits.append(logits.float().cpu())
        all_labels.append(lbls.float().cpu())
    
    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy()
    probs = 1 / (1 + np.exp(-all_logits))
    preds = (probs > 0.5).astype(int)
    
    # AUROC requires both classes present
    if len(np.unique(all_labels)) > 1:
        auroc = roc_auc_score(all_labels, probs)
    else:
        auroc = float('nan')
    
    acc = accuracy_score(all_labels, preds)
    
    return {
        'loss': total_loss / max(n_samples, 1),
        'acc': float(acc),
        'auroc': float(auroc),
        'n_samples': int(n_samples),
    }


def train_one_epoch(model, loader, optimizer, scheduler, criterion, ema, device,
                    grad_clip_norm, log_every=100):
    """
    One training epoch. Updates EMA every step.
    Returns average training loss and accuracy.
    """
    model.train()
    total_loss = 0.0
    correct = 0
    n_samples = 0
    
    pbar = tqdm(loader, desc='train', leave=False, ncols=100)
    for step, (imgs, lbls) in enumerate(pbar):
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(dtype=CFG.amp_dtype):
            logits = model(imgs)
            loss = criterion(logits, lbls)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()
        scheduler.step()
        
        # EMA update every step
        if ema is not None:
            ema.update(model)
        
        # Running stats
        with torch.no_grad():
            preds = (logits > 0).float()
            correct += (preds == lbls).sum().item()
            total_loss += loss.item() * lbls.size(0)
            n_samples += lbls.size(0)
        
        if step % log_every == 0:
            pbar.set_postfix({
                'loss': f'{total_loss / max(n_samples, 1):.4f}',
                'acc': f'{correct / max(n_samples, 1):.3f}',
                'lr_b': f'{optimizer.param_groups[0]["lr"]:.2e}',
            })
    
    return {
        'train_loss': total_loss / max(n_samples, 1),
        'train_acc': correct / max(n_samples, 1),
    }


# ============================================================
# Main per-model training function with smoothed BCE
# ============================================================
class SmoothBCEWithLogits(nn.Module):
    """
    Binary cross-entropy with label smoothing.
    
    Standard BCEWithLogitsLoss doesn't support label smoothing directly.
    Smoothing factor 0.05 means real labels become 0.05 and fake labels become 0.95.
    This prevents the model from getting overconfident on training data,
    which helps generalization across datasets.
    """
    def __init__(self, smoothing=0.05, pos_weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.pos_weight = pos_weight
    
    def forward(self, logits, targets):
        targets = targets.float()
        # Smooth: 0 -> smoothing, 1 -> 1-smoothing
        smooth_targets = targets * (1 - self.smoothing) + (1 - targets) * self.smoothing
        return F.binary_cross_entropy_with_logits(
            logits, smooth_targets,
            pos_weight=self.pos_weight,
        )


def train_one_model(
    model_name: str,
    model_builder,
    train_loader,
    val_loader,
    total_epochs: int,
    swa_window: int,
    backbone_lr: float,
    head_lr: float,
    warmup_epochs: int,
    holdout_name: str,
    use_compile: bool = False,
    save_dir: Path = None,
):
    """
    Train one model end-to-end with EMA, SWA, and full logging.
    
    Returns:
        dict with paths to saved checkpoints and final metrics
    
    Design notes:
    - The training schedule has 3 phases:
        1. Warmup epoch (1): backbone frozen, only head trains, LR linearly ramps
        2. Main training (epoch 2 to total-swa_window): full network, cosine decay
        3. SWA window (last swa_window epochs): same training, but accumulate weights
    - We track three candidate models at the end: regular final, EMA, SWA
    - We pick the best one based on validation AUROC and report which one won
    - Per-epoch metrics are appended to a JSONL file we can grep later
    """
    save_dir = save_dir or (LOO_CKPT_DIR / f"holdout_{holdout_name}")
    save_dir.mkdir(parents=True, exist_ok=True)
    
    log_dir = LOO_LOG_DIR / f"holdout_{holdout_name}"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{model_name}_epoch_log.jsonl"
    
    print(f"\n{'=' * 60}")
    print(f"  TRAINING {model_name.upper()} | holdout={holdout_name}")
    print(f"{'=' * 60}")
    print(f"  Total epochs : {total_epochs}")
    print(f"  Warmup       : {warmup_epochs} epoch(s)")
    print(f"  SWA window   : last {swa_window} epoch(s)")
    print(f"  Backbone LR  : {backbone_lr}")
    print(f"  Head LR      : {head_lr}")
    print(f"  torch.compile: {use_compile}")
    print(f"  Save dir     : {save_dir}")
    
    # ---- Build model ----
    model, backbone_params, head_params = model_builder()
    model = model.to(DEVICE)
    
    # Compile EFB4 only (we already verified the speedup)
    if use_compile:
        print(f"  Compiling model (this takes 30-60s on first call)...")
        model = torch.compile(model, mode='reduce-overhead', dynamic=False)
    
    # ---- Build optimizer + scheduler ----
    optimizer, scheduler = build_optimizer_and_scheduler(
        backbone_params, head_params,
        backbone_lr, head_lr,
        weight_decay=CFG.weight_decay,
        total_epochs=total_epochs,
        steps_per_epoch=len(train_loader),
        warmup_epochs=warmup_epochs,
    )
    
    # ---- Compute pos_weight from this holdout's training data ----
    # Critical for class balance — recomputed per holdout, not hardcoded
    train_labels = train_loader.dataset.labels
    n_real = int((train_labels == 0).sum())
    n_fake = int((train_labels == 1).sum())
    pos_weight = torch.tensor([n_real / max(n_fake, 1)], device=DEVICE)
    print(f"  Train balance: real={n_real:,}  fake={n_fake:,}  pos_weight={pos_weight.item():.3f}")
    
    criterion = SmoothBCEWithLogits(smoothing=CFG.label_smoothing, pos_weight=pos_weight)
    eval_criterion = nn.BCEWithLogitsLoss()  # unsmoothed for clean val numbers
    
    # ---- EMA + SWA trackers ----
    ema = CompileSafeEMA(model, decay=CFG.ema_decay)
    swa = ManualSWA()
    swa_start_epoch = total_epochs - swa_window  # 0-indexed
    
    # ---- Initial freeze for warmup ----
    if warmup_epochs > 0:
        freeze_backbone(backbone_params)
    
    # ---- Epoch loop ----
    best_val_auroc = -1.0
    best_epoch = -1
    history = []
    
    for epoch in range(total_epochs):
        epoch_start = time.time()
        
        # Unfreeze backbone after warmup
        if epoch == warmup_epochs and warmup_epochs > 0:
            print(f"  [epoch {epoch}] Unfreezing backbone")
            unfreeze_backbone(backbone_params)
        
        # Train
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, scheduler, criterion, ema,
            device=DEVICE, grad_clip_norm=CFG.grad_clip_norm,
        )
        
        # Validate (regular weights)
        val_metrics = evaluate_model(model, val_loader, eval_criterion, DEVICE, desc='val')
        
        # SWA update if we're in the window
        in_swa_window = epoch >= swa_start_epoch
        if in_swa_window:
            swa.update(model)
        
        epoch_time = time.time() - epoch_start
        gpu_mem = torch.cuda.max_memory_allocated() / 1e9
        torch.cuda.reset_peak_memory_stats()
        
        # Log row
        log_row = {
            'epoch': epoch,
            'train_loss': train_metrics['train_loss'],
            'train_acc': train_metrics['train_acc'],
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['acc'],
            'val_auroc': val_metrics['auroc'],
            'lr_backbone': optimizer.param_groups[0]['lr'],
            'lr_head': optimizer.param_groups[1]['lr'],
            'epoch_time_sec': epoch_time,
            'gpu_mem_peak_gb': gpu_mem,
            'in_swa_window': in_swa_window,
        }
        history.append(log_row)
        
        with open(log_path, 'a') as f:
            f.write(json.dumps(log_row) + '\n')
        
        print(f"  [epoch {epoch:2d}/{total_epochs-1}] "
              f"train_loss={train_metrics['train_loss']:.4f} "
              f"val_loss={val_metrics['loss']:.4f} "
              f"val_auroc={val_metrics['auroc']:.4f} "
              f"acc={val_metrics['acc']:.3f} "
              f"time={epoch_time:.0f}s "
              f"mem={gpu_mem:.1f}GB"
              f"{' [SWA]' if in_swa_window else ''}")
        
        # Track best regular checkpoint
        if val_metrics['auroc'] > best_val_auroc:
            best_val_auroc = val_metrics['auroc']
            best_epoch = epoch
            torch.save(
                ManualSWA._clean_state_dict(model.state_dict()),
                save_dir / f"{model_name}_best.pt"
            )
    
    # ---- End of training: save EMA and SWA ----
    torch.save(ema.state_dict(), save_dir / f"{model_name}_ema.pt")
    torch.save(swa.get_averaged_state_dict(), save_dir / f"{model_name}_swa.pt")
    
    print(f"\n  Training complete. Evaluating 3 candidates on val set...")
    
    # ---- Evaluate all 3 candidates on val to pick the winner ----
    candidates = {}
    
    # Regular best
    base_model_for_load, _, _ = model_builder()
    base_model_for_load = base_model_for_load.to(DEVICE)
    base_model_for_load.load_state_dict(torch.load(save_dir / f"{model_name}_best.pt"))
    candidates['best'] = evaluate_model(base_model_for_load, val_loader, eval_criterion, DEVICE)
    
    # EMA
    base_model_for_load.load_state_dict(ema.state_dict())
    candidates['ema'] = evaluate_model(base_model_for_load, val_loader, eval_criterion, DEVICE)
    
    # SWA — IMPORTANT: SWA changes BatchNorm running stats, need to recompute on training data
    base_model_for_load.load_state_dict(swa.get_averaged_state_dict())
    if any(isinstance(m, nn.BatchNorm2d) for m in base_model_for_load.modules()):
        print(f"  Recomputing BN stats for SWA model (1 pass over train data)...")
        recompute_bn_stats(base_model_for_load, train_loader, DEVICE, n_batches=200)
    candidates['swa'] = evaluate_model(base_model_for_load, val_loader, eval_criterion, DEVICE)
    # Save the BN-updated SWA weights
    torch.save(base_model_for_load.state_dict(), save_dir / f"{model_name}_swa.pt")
    
    print(f"\n  Candidate val AUROCs:")
    for name, m in candidates.items():
        print(f"    {name:6s}: AUROC={m['auroc']:.4f}  acc={m['acc']:.4f}")
    
    # Pick winner
    winner_name = max(candidates.keys(), key=lambda k: candidates[k]['auroc'])
    print(f"  Winner: {winner_name} (val AUROC={candidates[winner_name]['auroc']:.4f})")
    
    del model, base_model_for_load, ema, swa, optimizer, scheduler
    torch.cuda.empty_cache()
    
    return {
        'model_name': model_name,
        'holdout_name': holdout_name,
        'best_epoch': best_epoch,
        'history': history,
        'candidates': {k: {'auroc': v['auroc'], 'acc': v['acc']} for k, v in candidates.items()},
        'winner': winner_name,
        'winner_path': str(save_dir / f"{model_name}_{winner_name}.pt"),
        'all_paths': {
            'best': str(save_dir / f"{model_name}_best.pt"),
            'ema': str(save_dir / f"{model_name}_ema.pt"),
            'swa': str(save_dir / f"{model_name}_swa.pt"),
        },
        'log_path': str(log_path),
    }


@torch.no_grad()
def recompute_bn_stats(model, loader, device, n_batches=200):
    """
    SWA changes BN running stats — they're not actually averaged correctly because
    they're updated by forward passes during training, not by SWA's weight averaging.
    Standard fix: pass training data through the SWA model in train mode (with
    gradients off) for a few batches to recompute BN statistics.
    """
    model.train()  # train mode = BN updates running stats
    
    # Reset running stats
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.reset_running_stats()
            m.momentum = None  # use cumulative average instead of EMA
    
    n_done = 0
    for imgs, _ in loader:
        imgs = imgs.to(device, non_blocking=True)
        with autocast(dtype=CFG.amp_dtype):
            _ = model(imgs)
        n_done += 1
        if n_done >= n_batches:
            break
    
    model.eval()


print("Cell 4 loaded.")
print("Ready to test on one model. Run Cell 4b next to verify training works end-to-end.")

Cell 4 loaded.
Ready to test on one model. Run Cell 4b next to verify training works end-to-end.


In [16]:
# ============================================================
# Cell 4b: Quick smoke test of train_one_model on a tiny subset
# ============================================================

# Build a tiny version of the loaders for fast testing
tiny_train_df = loaders['train_df'].sample(2000, random_state=42).reset_index(drop=True)
tiny_val_df = loaders['val_df'].sample(500, random_state=42).reset_index(drop=True)

tiny_train_aug = build_train_augs(CFG.image_size, IMAGENET_MEAN, IMAGENET_STD)
tiny_val_aug = build_val_augs(CFG.image_size, IMAGENET_MEAN, IMAGENET_STD)

tiny_train_ds = DeepfakeDataset(tiny_train_df, tiny_train_aug)
tiny_val_ds = DeepfakeDataset(tiny_val_df, tiny_val_aug)

tiny_sampler = build_balanced_sampler(tiny_train_df)

tiny_train_loader = DataLoader(
    tiny_train_ds, batch_size=CFG.batch_size, sampler=tiny_sampler,
    num_workers=CFG.num_workers, pin_memory=True, persistent_workers=True,
    prefetch_factor=4, drop_last=True,
)
tiny_val_loader = DataLoader(
    tiny_val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=True, persistent_workers=True,
    prefetch_factor=4,
)

print(f"Tiny train: {len(tiny_train_ds)} samples, {len(tiny_train_loader)} batches")
print(f"Tiny val  : {len(tiny_val_ds)} samples, {len(tiny_val_loader)} batches")
print()

# Run a 2-epoch training to verify everything works
result = train_one_model(
    model_name='efb4',
    model_builder=build_efb4,
    train_loader=tiny_train_loader,
    val_loader=tiny_val_loader,
    total_epochs=2,
    swa_window=1,
    backbone_lr=CFG.efb4_backbone_lr,
    head_lr=CFG.efb4_head_lr,
    warmup_epochs=1,
    holdout_name='SMOKETEST',
    use_compile=True,
)

print(f"\n{'=' * 60}")
print("RESULT")
print('=' * 60)
print(json.dumps(result, indent=2, default=str))

Tiny train: 2000 samples, 20 batches
Tiny val  : 500 samples, 3 batches


  TRAINING EFB4 | holdout=SMOKETEST
  Total epochs : 2
  Warmup       : 1 epoch(s)
  SWA window   : last 1 epoch(s)
  Backbone LR  : 1e-05
  Head LR      : 0.001
  torch.compile: True
  Save dir     : /dataset/loo_sweep/checkpoints/holdout_SMOKETEST
  Compiling model (this takes 30-60s on first call)...
  Train balance: real=934  fake=1,066  pos_weight=0.876


train:   0%|                                                                 | 0/20 [00:01<?, ?it/s]

  [epoch  0/1] train_loss=2.7155 val_loss=1.8669 val_auroc=0.4705 acc=0.496 time=110s mem=32.9GB
  [epoch 1] Unfreezing backbone


train:   0%|                                                                 | 0/20 [00:00<?, ?it/s]

  [epoch  1/1] train_loss=2.2404 val_loss=1.7615 val_auroc=0.5054 acc=0.516 time=3s mem=2.6GB [SWA]

  Training complete. Evaluating 3 candidates on val set...
  Recomputing BN stats for SWA model (1 pass over train data)...

  Candidate val AUROCs:
    best  : AUROC=0.5034  acc=0.5200
    ema   : AUROC=0.4207  acc=0.4220
    swa   : AUROC=0.4970  acc=0.5160
  Winner: best (val AUROC=0.5034)

RESULT
{
  "model_name": "efb4",
  "holdout_name": "SMOKETEST",
  "best_epoch": 1,
  "history": [
    {
      "epoch": 0,
      "train_loss": 2.7155206918716432,
      "train_acc": 0.48125,
      "val_loss": 1.8669064655303955,
      "val_acc": 0.496,
      "val_auroc": 0.470528,
      "lr_backbone": 1e-05,
      "lr_head": 0.001,
      "epoch_time_sec": 109.85707497596741,
      "gpu_mem_peak_gb": 32.94093568,
      "in_swa_window": false
    },
    {
      "epoch": 1,
      "train_loss": 2.2404487133026123,
      "train_acc": 0.4921875,
      "val_loss": 1.7615465478897094,
      "val_acc": 0.51

In [17]:
# ============================================================
# Cell 5: Per-holdout orchestrator
# ============================================================
import pickle
from concurrent.futures import ThreadPoolExecutor


# ============================================================
# Helper: collect val predictions from a trained model
# (needed to fit the meta-learner)
# ============================================================
@torch.no_grad()
def collect_predictions(model, loader, device, return_indices=False):
    """
    Run model on a loader and return raw probabilities + labels (+ optionally indices).
    Used to gather predictions for meta-learner fitting and per-image logging.
    """
    model.eval()
    all_probs, all_labels, all_indices = [], [], []
    
    for batch in loader:
        if len(batch) == 3:
            imgs, lbls, idx = batch
            all_indices.append(idx.numpy())
        else:
            imgs, lbls = batch
        
        imgs = imgs.to(device, non_blocking=True)
        
        with autocast(dtype=CFG.amp_dtype):
            logits = model(imgs)
        
        probs = torch.sigmoid(logits.float()).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(lbls.numpy())
    
    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    
    if return_indices:
        indices = np.concatenate(all_indices) if all_indices else np.arange(len(labels))
        return probs, labels, indices
    return probs, labels


# ============================================================
# TTA inference — runs each image through 4 transforms and averages
# ============================================================
@torch.no_grad()
def tta_predict(model, df, normalize_mean, normalize_std, device):
    """
    Test-time augmentation: run the model 4 times per image with different
    deterministic transforms, average the probabilities. ~0.5-1.5 AUROC gain
    on cross-dataset eval, costs only inference compute.
    
    Returns:
        probs: [N] averaged probabilities
        labels: [N] true labels
        indices: [N] dataframe indices
    """
    model.eval()
    tta_transforms = build_tta_augs(CFG.image_size, normalize_mean, normalize_std)
    
    accumulated_probs = None
    final_labels = None
    final_indices = None
    
    for view_idx, transform in enumerate(tta_transforms):
        ds = DeepfakeDataset(df, transform, return_index=True)
        loader = DataLoader(
            ds, batch_size=CFG.batch_size * 2, shuffle=False,
            num_workers=CFG.num_workers, pin_memory=True,
            persistent_workers=False,  # one-shot, no need
        )
        
        probs, labels, indices = collect_predictions(model, loader, device, return_indices=True)
        
        if accumulated_probs is None:
            accumulated_probs = probs.copy()
            final_labels = labels
            final_indices = indices
        else:
            accumulated_probs += probs
        
        del ds, loader
    
    accumulated_probs /= len(tta_transforms)
    return accumulated_probs, final_labels, final_indices


# ============================================================
# Load a trained model from a checkpoint path
# ============================================================
def load_trained_model(model_builder, ckpt_path, device):
    """Build a fresh model and load weights from checkpoint."""
    model, _, _ = model_builder()
    state = torch.load(ckpt_path, map_location=device)
    # Strip _orig_mod. prefix if present (from compiled checkpoints)
    state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
    model.load_state_dict(state)
    model = model.to(device)
    model.eval()
    return model


# ============================================================
# Compute holdout-aware metrics
# ============================================================
def compute_holdout_metrics(probs, labels, holdout_name):
    """
    Compute metrics appropriate for the holdout type.
    
    For mixed-class holdouts (most): full AUROC, accuracy, F1, confusion matrix.
    For single-class holdouts (sd_generated, adversarial): only detection rate
    (recall on the fake class), since AUROC is undefined with one class.
    """
    is_single_class = holdout_name in SINGLE_CLASS_DATASETS
    preds = (probs > 0.5).astype(int)
    
    metrics = {
        'holdout_name': holdout_name,
        'n_samples': int(len(labels)),
        'n_real': int((labels == 0).sum()),
        'n_fake': int((labels == 1).sum()),
        'is_single_class': is_single_class,
        'mean_prob': float(probs.mean()),
        'std_prob': float(probs.std()),
    }
    
    if is_single_class:
        # All-fake holdout: detection rate = recall on the positive (fake) class
        metrics['detection_rate'] = float((preds == 1).mean())
        metrics['mean_fake_prob'] = float(probs.mean())
        metrics['auroc'] = None
        metrics['accuracy'] = None
        metrics['f1'] = None
    else:
        metrics['accuracy'] = float(accuracy_score(labels, preds))
        metrics['auroc'] = float(roc_auc_score(labels, probs))
        metrics['f1'] = float(f1_score(labels, preds))
        cm = confusion_matrix(labels, preds)
        metrics['tn'] = int(cm[0, 0])
        metrics['fp'] = int(cm[0, 1])
        metrics['fn'] = int(cm[1, 0])
        metrics['tp'] = int(cm[1, 1])
        metrics['precision'] = float(metrics['tp'] / max(metrics['tp'] + metrics['fp'], 1))
        metrics['recall'] = float(metrics['tp'] / max(metrics['tp'] + metrics['fn'], 1))
    
    return metrics


# ============================================================
# Main per-holdout function
# ============================================================
def run_one_holdout(holdout_name: str, parallel_swin_clip: bool = True):
    """
    Trains 3 models, fits ensemble meta-learner, evaluates on held-out test set
    with TTA, saves everything.
    
    Steps:
        1. Build dataloaders for this holdout
        2. Train EFB4 (compiled, alone — biggest model)
        3. Train Swin-T + CLIP (eager mode, in parallel via threads if requested)
        4. Load winning checkpoints, get val predictions for meta-learner
        5. Fit logistic regression meta-learner on val predictions
        6. Run TTA on held-out test set with all 3 models + ensemble
        7. Save per-image predictions and aggregate metrics
    """
    holdout_start = time.time()
    
    print(f"\n{'#' * 70}")
    print(f"# HOLDOUT: {holdout_name}")
    print(f"{'#' * 70}")
    
    # ---- Setup directories ----
    holdout_ckpt_dir = LOO_CKPT_DIR / f"holdout_{holdout_name}"
    holdout_ckpt_dir.mkdir(parents=True, exist_ok=True)
    
    holdout_results_dir = LOO_RESULTS_DIR / f"holdout_{holdout_name}"
    holdout_results_dir.mkdir(parents=True, exist_ok=True)
    
    # ---- Build loaders ----
    print(f"\n[1/6] Building dataloaders...")
    t0 = time.time()
    holdout_loaders = build_holdout_loaders(holdout_name)
    print(f"      train={len(holdout_loaders['train_df']):,} "
          f"val={len(holdout_loaders['val_df']):,} "
          f"test={len(holdout_loaders['test_df']):,}  "
          f"({time.time() - t0:.1f}s)")
    
    # ---- Train EFB4 ----
    print(f"\n[2/6] Training EFB4 (compiled)...")
    efb4_result = train_one_model(
        model_name='efb4',
        model_builder=build_efb4,
        train_loader=holdout_loaders['train_loader'],
        val_loader=holdout_loaders['val_loader'],
        total_epochs=CFG.efb4_total_epochs,
        swa_window=CFG.efb4_swa_window,
        backbone_lr=CFG.efb4_backbone_lr,
        head_lr=CFG.efb4_head_lr,
        warmup_epochs=CFG.efb4_warmup_epochs,
        holdout_name=holdout_name,
        use_compile=True,
    )
    torch.cuda.empty_cache()
    
    # ---- Train Swin-T and CLIP ----
    if parallel_swin_clip:
        print(f"\n[3/6] Training Swin-T + CLIP in parallel...")
        # Note: true parallel training on one GPU via threads is tricky because
        # PyTorch operations release the GIL but kernel launches still serialize
        # on the GPU. The actual speedup comes from interleaving data loading
        # and kernel execution. Realistic speedup: ~30%, not 2x.
        with ThreadPoolExecutor(max_workers=2) as executor:
            future_swin = executor.submit(
                train_one_model,
                model_name='swin_t',
                model_builder=build_swin_t,
                train_loader=holdout_loaders['train_loader'],
                val_loader=holdout_loaders['val_loader'],
                total_epochs=CFG.swin_total_epochs,
                swa_window=CFG.swin_swa_window,
                backbone_lr=CFG.swin_backbone_lr,
                head_lr=CFG.swin_head_lr,
                warmup_epochs=CFG.swin_warmup_epochs,
                holdout_name=holdout_name,
                use_compile=False,
            )
            future_clip = executor.submit(
                train_one_model,
                model_name='clip_vit',
                model_builder=build_clip_vit_b16,
                train_loader=holdout_loaders['train_loader'],
                val_loader=holdout_loaders['val_loader'],
                total_epochs=CFG.clip_total_epochs,
                swa_window=CFG.clip_swa_window,
                backbone_lr=CFG.clip_backbone_lr,
                head_lr=CFG.clip_head_lr,
                warmup_epochs=CFG.clip_warmup_epochs,
                holdout_name=holdout_name,
                use_compile=False,
            )
            swin_result = future_swin.result()
            clip_result = future_clip.result()
    else:
        print(f"\n[3/6] Training Swin-T...")
        swin_result = train_one_model(
            model_name='swin_t', model_builder=build_swin_t,
            train_loader=holdout_loaders['train_loader'],
            val_loader=holdout_loaders['val_loader'],
            total_epochs=CFG.swin_total_epochs, swa_window=CFG.swin_swa_window,
            backbone_lr=CFG.swin_backbone_lr, head_lr=CFG.swin_head_lr,
            warmup_epochs=CFG.swin_warmup_epochs, holdout_name=holdout_name,
            use_compile=False,
        )
        torch.cuda.empty_cache()
        
        print(f"\n      Training CLIP ViT-B/16...")
        clip_result = train_one_model(
            model_name='clip_vit', model_builder=build_clip_vit_b16,
            train_loader=holdout_loaders['train_loader'],
            val_loader=holdout_loaders['val_loader'],
            total_epochs=CFG.clip_total_epochs, swa_window=CFG.clip_swa_window,
            backbone_lr=CFG.clip_backbone_lr, head_lr=CFG.clip_head_lr,
            warmup_epochs=CFG.clip_warmup_epochs, holdout_name=holdout_name,
            use_compile=False,
        )
    
    torch.cuda.empty_cache()
    
    # ---- Fit meta-learner on val predictions ----
    print(f"\n[4/6] Collecting val predictions for meta-learner...")
    
    # Load each model's WINNING checkpoint and get val probabilities
    model_results = {'efb4': efb4_result, 'swin_t': swin_result, 'clip_vit': clip_result}
    model_builders = {'efb4': build_efb4, 'swin_t': build_swin_t, 'clip_vit': build_clip_vit_b16}
    val_probs_per_model = {}
    val_labels_ref = None
    
    for mname, mres in model_results.items():
        winner_path = mres['winner_path']
        print(f"      Loading {mname} ({mres['winner']}): {Path(winner_path).name}")
        m = load_trained_model(model_builders[mname], winner_path, DEVICE)
        probs, labels = collect_predictions(m, holdout_loaders['val_loader'], DEVICE)
        val_probs_per_model[mname] = probs
        if val_labels_ref is None:
            val_labels_ref = labels
        else:
            assert np.array_equal(labels, val_labels_ref), "Val label mismatch across models"
        del m
        torch.cuda.empty_cache()
    
    # Stack into [N, 3] feature matrix for the meta-learner
    val_features = np.stack([
        val_probs_per_model['efb4'],
        val_probs_per_model['swin_t'],
        val_probs_per_model['clip_vit'],
    ], axis=1)
    
    print(f"      Fitting logistic regression meta-learner on {len(val_labels_ref):,} val samples...")
    meta_learner = LogisticRegression(
        C=1.0,                   # L2 regularization strength
        max_iter=1000,
        class_weight='balanced',  # in case val isn't perfectly balanced
        solver='lbfgs',
        random_state=CFG.seed,
    )
    meta_learner.fit(val_features, val_labels_ref)
    
    # Report meta-learner weights — interpretability
    coefs = meta_learner.coef_[0]
    print(f"      Meta-learner coefficients (higher = more weight):")
    print(f"        EFB4    : {coefs[0]:+.3f}")
    print(f"        Swin-T  : {coefs[1]:+.3f}")
    print(f"        CLIP ViT: {coefs[2]:+.3f}")
    print(f"        intercept: {meta_learner.intercept_[0]:+.3f}")
    
    # Sanity check meta-learner on val
    val_ensemble_probs = meta_learner.predict_proba(val_features)[:, 1]
    val_ensemble_auroc = roc_auc_score(val_labels_ref, val_ensemble_probs)
    print(f"      Meta-learner val AUROC: {val_ensemble_auroc:.4f}")
    
    # Save meta-learner
    meta_path = holdout_ckpt_dir / "meta_learner.pkl"
    with open(meta_path, 'wb') as f:
        pickle.dump(meta_learner, f)
    
    # ---- TTA evaluation on held-out test set ----
    print(f"\n[5/6] TTA inference on held-out test set ({len(holdout_loaders['test_df']):,} images)...")
    
    test_df = holdout_loaders['test_df']
    test_probs_per_model = {}
    test_labels_ref = None
    test_indices_ref = None
    
    for mname, mres in model_results.items():
        winner_path = mres['winner_path']
        print(f"      TTA inference: {mname}")
        m = load_trained_model(model_builders[mname], winner_path, DEVICE)
        probs, labels, indices = tta_predict(m, test_df, IMAGENET_MEAN, IMAGENET_STD, DEVICE)
        test_probs_per_model[mname] = probs
        if test_labels_ref is None:
            test_labels_ref = labels
            test_indices_ref = indices
        del m
        torch.cuda.empty_cache()
    
    # Build test feature matrix and apply meta-learner
    test_features = np.stack([
        test_probs_per_model['efb4'],
        test_probs_per_model['swin_t'],
        test_probs_per_model['clip_vit'],
    ], axis=1)
    test_ensemble_probs = meta_learner.predict_proba(test_features)[:, 1]
    
    # ---- Compute and save metrics ----
    print(f"\n[6/6] Computing metrics and saving...")
    
    per_model_metrics = {}
    for mname, probs in test_probs_per_model.items():
        per_model_metrics[mname] = compute_holdout_metrics(probs, test_labels_ref, holdout_name)
    
    ensemble_metrics = compute_holdout_metrics(test_ensemble_probs, test_labels_ref, holdout_name)
    
    print(f"\n      RESULTS for holdout '{holdout_name}':")
    if holdout_name in SINGLE_CLASS_DATASETS:
        print(f"        (Single-class dataset — reporting detection rate)")
        for mname, m in per_model_metrics.items():
            print(f"        {mname:10s}: detection_rate={m['detection_rate']:.4f}")
        print(f"        ENSEMBLE  : detection_rate={ensemble_metrics['detection_rate']:.4f}")
    else:
        for mname, m in per_model_metrics.items():
            print(f"        {mname:10s}: AUROC={m['auroc']:.4f}  acc={m['accuracy']:.4f}  f1={m['f1']:.4f}")
        print(f"        ENSEMBLE  : AUROC={ensemble_metrics['auroc']:.4f}  "
              f"acc={ensemble_metrics['accuracy']:.4f}  f1={ensemble_metrics['f1']:.4f}")
    
    # ---- Save per-image predictions as parquet ----
    per_image_df = pd.DataFrame({
        'idx': test_indices_ref,
        'path': [test_df['path'].iloc[i] for i in test_indices_ref],
        'true_label': test_labels_ref.astype(int),
        'efb4_prob': test_probs_per_model['efb4'],
        'swin_t_prob': test_probs_per_model['swin_t'],
        'clip_vit_prob': test_probs_per_model['clip_vit'],
        'ensemble_prob': test_ensemble_probs,
    })
    parquet_path = holdout_results_dir / "per_image_predictions.parquet"
    per_image_df.to_parquet(parquet_path, index=False)
    print(f"      Saved per-image predictions: {parquet_path.name} ({len(per_image_df):,} rows)")
    
    # ---- Save aggregate metrics JSON ----
    holdout_total_time = time.time() - holdout_start
    
    holdout_summary = {
        'holdout_name': holdout_name,
        'total_time_sec': holdout_total_time,
        'total_time_min': holdout_total_time / 60,
        'train_size': int(len(holdout_loaders['train_df'])),
        'val_size': int(len(holdout_loaders['val_df'])),
        'test_size': int(len(holdout_loaders['test_df'])),
        'per_model_metrics': per_model_metrics,
        'ensemble_metrics': ensemble_metrics,
        'meta_learner': {
            'coefficients': coefs.tolist(),
            'intercept': float(meta_learner.intercept_[0]),
            'val_auroc': float(val_ensemble_auroc),
        },
        'model_winners': {
            'efb4': efb4_result['winner'],
            'swin_t': swin_result['winner'],
            'clip_vit': clip_result['winner'],
        },
        'training_results': {
            'efb4': {
                'best_epoch': efb4_result['best_epoch'],
                'candidates': efb4_result['candidates'],
            },
            'swin_t': {
                'best_epoch': swin_result['best_epoch'],
                'candidates': swin_result['candidates'],
            },
            'clip_vit': {
                'best_epoch': clip_result['best_epoch'],
                'candidates': clip_result['candidates'],
            },
        },
    }
    
    summary_path = holdout_results_dir / "summary.json"
    with open(summary_path, 'w') as f:
        json.dump(holdout_summary, f, indent=2)
    print(f"      Saved summary: {summary_path.name}")
    
    print(f"\n      Holdout '{holdout_name}' complete in {holdout_total_time / 60:.1f} min")
    print(f"{'#' * 70}\n")
    
    return holdout_summary


print("Cell 5 loaded.")
print("\nReady to run a full real holdout. The next cell will run real_and_fake_face_clean_v1")
print("end-to-end as our benchmark holdout.")

Cell 5 loaded.

Ready to run a full real holdout. The next cell will run real_and_fake_face_clean_v1
end-to-end as our benchmark holdout.


In [18]:
# ============================================================
# Cell 5b: Run the benchmark holdout (real_and_fake_face_clean_v1)
# ============================================================
benchmark_summary = run_one_holdout(
    holdout_name='real_and_fake_face_clean_v1',
    parallel_swin_clip=True,
)

print("\n" + "=" * 70)
print("BENCHMARK HOLDOUT COMPLETE")
print("=" * 70)
print(f"Total time: {benchmark_summary['total_time_min']:.1f} min")
print(f"\nIf this looks correct, the full 8-holdout sweep estimate is:")
print(f"  Sequential : {benchmark_summary['total_time_min'] * 8 / 60:.1f} hours")
print(f"  Cost @ $1.75/hr: ${benchmark_summary['total_time_min'] * 8 / 60 * 1.75:.2f}")


######################################################################
# HOLDOUT: real_and_fake_face_clean_v1
######################################################################

[1/6] Building dataloaders...
      train=341,649 val=85,414 test=1,680  (0.6s)

[2/6] Training EFB4 (compiled)...

  TRAINING EFB4 | holdout=real_and_fake_face_clean_v1
  Total epochs : 18
  Warmup       : 1 epoch(s)
  SWA window   : last 5 epoch(s)
  Backbone LR  : 1e-05
  Head LR      : 0.001
  torch.compile: True
  Save dir     : /dataset/loo_sweep/checkpoints/holdout_real_and_fake_face_clean_v1
  Compiling model (this takes 30-60s on first call)...
  Train balance: real=162,807  fake=178,842  pos_weight=0.910


train:   0%|                                                               | 0/3558 [00:01<?, ?it/s]

  [epoch  0/17] train_loss=1.8296 val_loss=0.9174 val_auroc=0.5712 acc=0.552 time=162s mem=5.2GB
  [epoch 1] Unfreezing backbone


train:   0%|                                                               | 0/3558 [00:00<?, ?it/s]

  [epoch  1/17] train_loss=0.8922 val_loss=0.6842 val_auroc=0.6135 acc=0.578 time=148s mem=3.2GB


...

  [epoch  8/9] train_loss=0.4742 val_loss=0.4488 val_auroc=0.8749 acc=0.788 time=63s mem=12.9GB [SWA]


train:   0%|                                                               | 0/3558 [00:00<?, ?it/s]

  [epoch  8/9] train_loss=0.3019 val_loss=0.1749 val_auroc=0.9866 acc=0.945 time=62s mem=11.7GB [SWA]


train:   0%|                                                               | 0/3558 [00:00<?, ?it/s]

  [epoch  9/9] train_loss=0.4628 val_loss=0.4600 val_auroc=0.8955 acc=0.774 time=64s mem=13.0GB [SWA]

  Training complete. Evaluating 3 candidates on val set...

  Candidate val AUROCs:
    best  : AUROC=0.8955  acc=0.7743
    ema   : AUROC=0.5783  acc=0.5571
    swa   : AUROC=0.8828  acc=0.7911
  Winner: best (val AUROC=0.8955)
  [epoch  9/9] train_loss=0.2791 val_loss=0.1428 val_auroc=0.9929 acc=0.959 time=293s mem=13.1GB [SWA]

  Training complete. Evaluating 3 candidates on val set...

  Candidate val AUROCs:
    best  : AUROC=0.9929  acc=0.9588
    ema   : AUROC=0.9168  acc=0.7010
    swa   : AUROC=0.9942  acc=0.9614
  Winner: swa (val AUROC=0.9942)

[4/6] Collecting val predictions for meta-learner...
      Loading efb4 (best): efb4_best.pt
      Loading swin_t (best): swin_t_best.pt
      Loading clip_vit (swa): clip_vit_swa.pt
      Fitting logistic regression meta-learner on 85,414 val samples...
      Meta-learner coefficients (higher = more weight):
        EFB4    : -2.910

In [19]:
# ============================================================
# Cell 5c: Cleanup patch — drop EFB4, drop EMA, drop kaggle dataset
# ============================================================

# ---- 1. Drop real_and_fake_face_clean_v1 from master ----
DROPPED_DATASET = "real_and_fake_face_clean_v1"

original_size = len(MASTER_DF)
MASTER_DF = MASTER_DF[MASTER_DF['dataset'] != DROPPED_DATASET].reset_index(drop=True)
new_size = len(MASTER_DF)
print(f"Dropped {DROPPED_DATASET}: {original_size:,} -> {new_size:,} rows ({original_size - new_size:,} removed)")

# Update the dataset list
ALL_DATASETS = [d for d in ALL_DATASETS if d != DROPPED_DATASET]
print(f"Active datasets ({len(ALL_DATASETS)}): {ALL_DATASETS}")

print(f"\nUpdated per-dataset counts:")
print(MASTER_DF['dataset'].value_counts().to_string())
print(f"\nUpdated label distribution:")
print(MASTER_DF['label'].value_counts().to_string())


# ============================================================
# 2. New train_one_model — EMA removed, SWA only
# ============================================================
def train_one_model_v2(
    model_name: str,
    model_builder,
    train_loader,
    val_loader,
    total_epochs: int,
    swa_window: int,
    backbone_lr: float,
    head_lr: float,
    warmup_epochs: int,
    holdout_name: str,
    use_compile: bool = False,
    save_dir: Path = None,
):
    """
    Trains one model with SWA only (no EMA). Otherwise identical to v1.
    
    Changes from v1:
    - Removed EMA tracker entirely (was buggy and SWA dominated anyway)
    - Candidate evaluation now compares only 'best' vs 'swa'
    - Saved checkpoints: <name>_best.pt and <name>_swa.pt only
    """
    save_dir = save_dir or (LOO_CKPT_DIR / f"holdout_{holdout_name}")
    save_dir.mkdir(parents=True, exist_ok=True)
    
    log_dir = LOO_LOG_DIR / f"holdout_{holdout_name}"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{model_name}_epoch_log.jsonl"
    
    print(f"\n{'=' * 60}")
    print(f"  TRAINING {model_name.upper()} | holdout={holdout_name}")
    print(f"{'=' * 60}")
    print(f"  Total epochs : {total_epochs}")
    print(f"  Warmup       : {warmup_epochs} epoch(s)")
    print(f"  SWA window   : last {swa_window} epoch(s)")
    print(f"  Backbone LR  : {backbone_lr}")
    print(f"  Head LR      : {head_lr}")
    print(f"  torch.compile: {use_compile}")
    print(f"  Save dir     : {save_dir}")
    
    model, backbone_params, head_params = model_builder()
    model = model.to(DEVICE)
    
    if use_compile:
        print(f"  Compiling model (this takes 30-60s on first call)...")
        model = torch.compile(model, mode='reduce-overhead', dynamic=False)
    
    optimizer, scheduler = build_optimizer_and_scheduler(
        backbone_params, head_params,
        backbone_lr, head_lr,
        weight_decay=CFG.weight_decay,
        total_epochs=total_epochs,
        steps_per_epoch=len(train_loader),
        warmup_epochs=warmup_epochs,
    )
    
    # Per-holdout pos_weight
    train_labels = train_loader.dataset.labels
    n_real = int((train_labels == 0).sum())
    n_fake = int((train_labels == 1).sum())
    pos_weight = torch.tensor([n_real / max(n_fake, 1)], device=DEVICE)
    print(f"  Train balance: real={n_real:,}  fake={n_fake:,}  pos_weight={pos_weight.item():.3f}")
    
    criterion = SmoothBCEWithLogits(smoothing=CFG.label_smoothing, pos_weight=pos_weight)
    eval_criterion = nn.BCEWithLogitsLoss()
    
    # SWA only — no EMA
    swa = ManualSWA()
    swa_start_epoch = total_epochs - swa_window
    
    if warmup_epochs > 0:
        freeze_backbone(backbone_params)
    
    best_val_auroc = -1.0
    best_epoch = -1
    history = []
    
    for epoch in range(total_epochs):
        epoch_start = time.time()
        
        if epoch == warmup_epochs and warmup_epochs > 0:
            print(f"  [epoch {epoch}] Unfreezing backbone")
            unfreeze_backbone(backbone_params)
        
        # Train (passing ema=None disables EMA updates in train_one_epoch)
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, scheduler, criterion, ema=None,
            device=DEVICE, grad_clip_norm=CFG.grad_clip_norm,
        )
        
        val_metrics = evaluate_model(model, val_loader, eval_criterion, DEVICE, desc='val')
        
        in_swa_window = epoch >= swa_start_epoch
        if in_swa_window:
            swa.update(model)
        
        epoch_time = time.time() - epoch_start
        gpu_mem = torch.cuda.max_memory_allocated() / 1e9
        torch.cuda.reset_peak_memory_stats()
        
        log_row = {
            'epoch': epoch,
            'train_loss': train_metrics['train_loss'],
            'train_acc': train_metrics['train_acc'],
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['acc'],
            'val_auroc': val_metrics['auroc'],
            'lr_backbone': optimizer.param_groups[0]['lr'],
            'lr_head': optimizer.param_groups[1]['lr'],
            'epoch_time_sec': epoch_time,
            'gpu_mem_peak_gb': gpu_mem,
            'in_swa_window': in_swa_window,
        }
        history.append(log_row)
        
        with open(log_path, 'a') as f:
            f.write(json.dumps(log_row) + '\n')
        
        print(f"  [epoch {epoch:2d}/{total_epochs-1}] "
              f"train_loss={train_metrics['train_loss']:.4f} "
              f"val_loss={val_metrics['loss']:.4f} "
              f"val_auroc={val_metrics['auroc']:.4f} "
              f"acc={val_metrics['acc']:.3f} "
              f"time={epoch_time:.0f}s "
              f"mem={gpu_mem:.1f}GB"
              f"{' [SWA]' if in_swa_window else ''}")
        
        if val_metrics['auroc'] > best_val_auroc:
            best_val_auroc = val_metrics['auroc']
            best_epoch = epoch
            torch.save(
                ManualSWA._clean_state_dict(model.state_dict()),
                save_dir / f"{model_name}_best.pt"
            )
    
    # Save SWA
    torch.save(swa.get_averaged_state_dict(), save_dir / f"{model_name}_swa.pt")
    
    print(f"\n  Training complete. Evaluating 2 candidates on val set...")
    
    # ---- Evaluate 2 candidates ----
    candidates = {}
    
    base_for_load, _, _ = model_builder()
    base_for_load = base_for_load.to(DEVICE)
    
    base_for_load.load_state_dict(torch.load(save_dir / f"{model_name}_best.pt"))
    candidates['best'] = evaluate_model(base_for_load, val_loader, eval_criterion, DEVICE)
    
    base_for_load.load_state_dict(swa.get_averaged_state_dict())
    if any(isinstance(m, nn.BatchNorm2d) for m in base_for_load.modules()):
        print(f"  Recomputing BN stats for SWA model...")
        recompute_bn_stats(base_for_load, train_loader, DEVICE, n_batches=200)
    candidates['swa'] = evaluate_model(base_for_load, val_loader, eval_criterion, DEVICE)
    torch.save(base_for_load.state_dict(), save_dir / f"{model_name}_swa.pt")
    
    print(f"\n  Candidate val AUROCs:")
    for name, m in candidates.items():
        print(f"    {name:6s}: AUROC={m['auroc']:.4f}  acc={m['acc']:.4f}")
    
    winner_name = max(candidates.keys(), key=lambda k: candidates[k]['auroc'])
    print(f"  Winner: {winner_name} (val AUROC={candidates[winner_name]['auroc']:.4f})")
    
    del model, base_for_load, swa, optimizer, scheduler
    torch.cuda.empty_cache()
    
    return {
        'model_name': model_name,
        'holdout_name': holdout_name,
        'best_epoch': best_epoch,
        'history': history,
        'candidates': {k: {'auroc': v['auroc'], 'acc': v['acc']} for k, v in candidates.items()},
        'winner': winner_name,
        'winner_path': str(save_dir / f"{model_name}_{winner_name}.pt"),
        'all_paths': {
            'best': str(save_dir / f"{model_name}_best.pt"),
            'swa': str(save_dir / f"{model_name}_swa.pt"),
        },
        'log_path': str(log_path),
    }


# ============================================================
# 3. New run_one_holdout — CLIP + Swin only, no EFB4
# ============================================================
def run_one_holdout_v2(holdout_name: str, parallel: bool = True):
    """
    Trains CLIP + Swin (parallel), fits meta-learner, evaluates with TTA.
    Drops EFB4 from the original v1 orchestrator.
    """
    holdout_start = time.time()
    
    print(f"\n{'#' * 70}")
    print(f"# HOLDOUT: {holdout_name}  (CLIP + Swin only)")
    print(f"{'#' * 70}")
    
    holdout_ckpt_dir = LOO_CKPT_DIR / f"holdout_{holdout_name}"
    holdout_ckpt_dir.mkdir(parents=True, exist_ok=True)
    holdout_results_dir = LOO_RESULTS_DIR / f"holdout_{holdout_name}"
    holdout_results_dir.mkdir(parents=True, exist_ok=True)
    
    # ---- Build loaders ----
    print(f"\n[1/5] Building dataloaders...")
    t0 = time.time()
    holdout_loaders = build_holdout_loaders(holdout_name)
    print(f"      train={len(holdout_loaders['train_df']):,} "
          f"val={len(holdout_loaders['val_df']):,} "
          f"test={len(holdout_loaders['test_df']):,}  "
          f"({time.time() - t0:.1f}s)")
    
    # ---- Train Swin + CLIP (parallel or sequential) ----
    if parallel:
        print(f"\n[2/5] Training Swin-T + CLIP in parallel...")
        with ThreadPoolExecutor(max_workers=2) as executor:
            future_swin = executor.submit(
                train_one_model_v2,
                model_name='swin_t', model_builder=build_swin_t,
                train_loader=holdout_loaders['train_loader'],
                val_loader=holdout_loaders['val_loader'],
                total_epochs=CFG.swin_total_epochs, swa_window=CFG.swin_swa_window,
                backbone_lr=CFG.swin_backbone_lr, head_lr=CFG.swin_head_lr,
                warmup_epochs=CFG.swin_warmup_epochs, holdout_name=holdout_name,
                use_compile=False,
            )
            future_clip = executor.submit(
                train_one_model_v2,
                model_name='clip_vit', model_builder=build_clip_vit_b16,
                train_loader=holdout_loaders['train_loader'],
                val_loader=holdout_loaders['val_loader'],
                total_epochs=CFG.clip_total_epochs, swa_window=CFG.clip_swa_window,
                backbone_lr=CFG.clip_backbone_lr, head_lr=CFG.clip_head_lr,
                warmup_epochs=CFG.clip_warmup_epochs, holdout_name=holdout_name,
                use_compile=False,
            )
            swin_result = future_swin.result()
            clip_result = future_clip.result()
    else:
        print(f"\n[2/5] Training Swin-T...")
        swin_result = train_one_model_v2(
            model_name='swin_t', model_builder=build_swin_t,
            train_loader=holdout_loaders['train_loader'],
            val_loader=holdout_loaders['val_loader'],
            total_epochs=CFG.swin_total_epochs, swa_window=CFG.swin_swa_window,
            backbone_lr=CFG.swin_backbone_lr, head_lr=CFG.swin_head_lr,
            warmup_epochs=CFG.swin_warmup_epochs, holdout_name=holdout_name,
            use_compile=False,
        )
        torch.cuda.empty_cache()
        print(f"\n      Training CLIP ViT-B/16...")
        clip_result = train_one_model_v2(
            model_name='clip_vit', model_builder=build_clip_vit_b16,
            train_loader=holdout_loaders['train_loader'],
            val_loader=holdout_loaders['val_loader'],
            total_epochs=CFG.clip_total_epochs, swa_window=CFG.clip_swa_window,
            backbone_lr=CFG.clip_backbone_lr, head_lr=CFG.clip_head_lr,
            warmup_epochs=CFG.clip_warmup_epochs, holdout_name=holdout_name,
            use_compile=False,
        )
    
    torch.cuda.empty_cache()
    
    # ---- Fit meta-learner ----
    print(f"\n[3/5] Collecting val predictions for meta-learner...")
    
    model_results = {'swin_t': swin_result, 'clip_vit': clip_result}
    model_builders = {'swin_t': build_swin_t, 'clip_vit': build_clip_vit_b16}
    val_probs_per_model = {}
    val_labels_ref = None
    
    for mname, mres in model_results.items():
        winner_path = mres['winner_path']
        print(f"      Loading {mname} ({mres['winner']}): {Path(winner_path).name}")
        m = load_trained_model(model_builders[mname], winner_path, DEVICE)
        probs, labels = collect_predictions(m, holdout_loaders['val_loader'], DEVICE)
        val_probs_per_model[mname] = probs
        if val_labels_ref is None:
            val_labels_ref = labels
        del m
        torch.cuda.empty_cache()
    
    val_features = np.stack([
        val_probs_per_model['swin_t'],
        val_probs_per_model['clip_vit'],
    ], axis=1)
    
    print(f"      Fitting logistic regression meta-learner on {len(val_labels_ref):,} val samples...")
    meta_learner = LogisticRegression(
        C=1.0, max_iter=1000, class_weight='balanced',
        solver='lbfgs', random_state=CFG.seed,
    )
    meta_learner.fit(val_features, val_labels_ref)
    
    coefs = meta_learner.coef_[0]
    print(f"      Meta-learner coefficients:")
    print(f"        Swin-T  : {coefs[0]:+.3f}")
    print(f"        CLIP ViT: {coefs[1]:+.3f}")
    print(f"        intercept: {meta_learner.intercept_[0]:+.3f}")
    
    val_ensemble_probs = meta_learner.predict_proba(val_features)[:, 1]
    val_ensemble_auroc = roc_auc_score(val_labels_ref, val_ensemble_probs)
    print(f"      Meta-learner val AUROC: {val_ensemble_auroc:.4f}")
    
    meta_path = holdout_ckpt_dir / "meta_learner.pkl"
    with open(meta_path, 'wb') as f:
        pickle.dump(meta_learner, f)
    
    # ---- TTA evaluation ----
    print(f"\n[4/5] TTA inference on held-out test set ({len(holdout_loaders['test_df']):,} images)...")
    
    test_df = holdout_loaders['test_df']
    test_probs_per_model = {}
    test_labels_ref = None
    test_indices_ref = None
    
    for mname, mres in model_results.items():
        winner_path = mres['winner_path']
        print(f"      TTA inference: {mname}")
        m = load_trained_model(model_builders[mname], winner_path, DEVICE)
        probs, labels, indices = tta_predict(m, test_df, IMAGENET_MEAN, IMAGENET_STD, DEVICE)
        test_probs_per_model[mname] = probs
        if test_labels_ref is None:
            test_labels_ref = labels
            test_indices_ref = indices
        del m
        torch.cuda.empty_cache()
    
    test_features = np.stack([
        test_probs_per_model['swin_t'],
        test_probs_per_model['clip_vit'],
    ], axis=1)
    test_ensemble_probs = meta_learner.predict_proba(test_features)[:, 1]
    
    # ---- Save metrics ----
    print(f"\n[5/5] Computing metrics and saving...")
    
    per_model_metrics = {}
    for mname, probs in test_probs_per_model.items():
        per_model_metrics[mname] = compute_holdout_metrics(probs, test_labels_ref, holdout_name)
    ensemble_metrics = compute_holdout_metrics(test_ensemble_probs, test_labels_ref, holdout_name)
    
    print(f"\n      RESULTS for holdout '{holdout_name}':")
    if holdout_name in SINGLE_CLASS_DATASETS:
        print(f"        (Single-class dataset — reporting detection rate)")
        for mname, m in per_model_metrics.items():
            print(f"        {mname:10s}: detection_rate={m['detection_rate']:.4f}")
        print(f"        ENSEMBLE  : detection_rate={ensemble_metrics['detection_rate']:.4f}")
    else:
        for mname, m in per_model_metrics.items():
            print(f"        {mname:10s}: AUROC={m['auroc']:.4f}  acc={m['accuracy']:.4f}  f1={m['f1']:.4f}")
        print(f"        ENSEMBLE  : AUROC={ensemble_metrics['auroc']:.4f}  "
              f"acc={ensemble_metrics['accuracy']:.4f}  f1={ensemble_metrics['f1']:.4f}")
    
    # Save per-image
    per_image_df = pd.DataFrame({
        'idx': test_indices_ref,
        'path': [test_df['path'].iloc[i] for i in test_indices_ref],
        'true_label': test_labels_ref.astype(int),
        'swin_t_prob': test_probs_per_model['swin_t'],
        'clip_vit_prob': test_probs_per_model['clip_vit'],
        'ensemble_prob': test_ensemble_probs,
    })
    parquet_path = holdout_results_dir / "per_image_predictions.parquet"
    per_image_df.to_parquet(parquet_path, index=False)
    print(f"      Saved per-image predictions: {parquet_path.name} ({len(per_image_df):,} rows)")
    
    holdout_total_time = time.time() - holdout_start
    
    holdout_summary = {
        'holdout_name': holdout_name,
        'total_time_sec': holdout_total_time,
        'total_time_min': holdout_total_time / 60,
        'train_size': int(len(holdout_loaders['train_df'])),
        'val_size': int(len(holdout_loaders['val_df'])),
        'test_size': int(len(holdout_loaders['test_df'])),
        'per_model_metrics': per_model_metrics,
        'ensemble_metrics': ensemble_metrics,
        'meta_learner': {
            'coefficients': coefs.tolist(),
            'intercept': float(meta_learner.intercept_[0]),
            'val_auroc': float(val_ensemble_auroc),
        },
        'model_winners': {
            'swin_t': swin_result['winner'],
            'clip_vit': clip_result['winner'],
        },
        'training_results': {
            'swin_t': {
                'best_epoch': swin_result['best_epoch'],
                'candidates': swin_result['candidates'],
            },
            'clip_vit': {
                'best_epoch': clip_result['best_epoch'],
                'candidates': clip_result['candidates'],
            },
        },
    }
    
    summary_path = holdout_results_dir / "summary.json"
    with open(summary_path, 'w') as f:
        json.dump(holdout_summary, f, indent=2)
    print(f"      Saved summary: {summary_path.name}")
    
    print(f"\n      Holdout '{holdout_name}' complete in {holdout_total_time / 60:.1f} min")
    print(f"{'#' * 70}\n")
    
    return holdout_summary


print("\nPatch applied. Ready to run validation holdout on celeb_df_v2.")
print(f"  Active datasets: {len(ALL_DATASETS)}")
print(f"  Models per holdout: 2 (CLIP ViT-B/16 + Swin-T)")
print(f"  Tracking: SWA only (EMA removed)")

Dropped real_and_fake_face_clean_v1: 428,743 -> 427,063 rows (1,680 removed)
Active datasets (7): ['deepfake_and_real_clean_v1', '140k_clean_v1', 'artifact_clean_v2', 'ffpp_c23', 'celeb_df_v2', 'sd_generated', 'adversarial']

Updated per-dataset counts:
dataset
deepfake_and_real_clean_v1    142528
140k_clean_v1                 123056
artifact_clean_v2              78420
ffpp_c23                       38309
celeb_df_v2                    24750
sd_generated                   10000
adversarial                    10000

Updated label distribution:
label
1    223553
0    203510

Patch applied. Ready to run validation holdout on celeb_df_v2.
  Active datasets: 7
  Models per holdout: 2 (CLIP ViT-B/16 + Swin-T)
  Tracking: SWA only (EMA removed)
